# Notebook 8 — Active glued growth-rate sweep for selected $D_\theta$ regimes

## 0. Imports, paths, and run controls

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from front_runner import compile_c_code, run_front_simulations
from front_analysis import (
    read_front_file,
    get_run_files,
    fit_front_speeds_for_front_data,
    side_speed_long_table,
)

project_folder = Path('.').resolve()

c_file = project_folder / 'abm_front_propagation_glued.c'
exe_file = project_folder / 'abm_front_propagation_glued_active_growth.exe'

param_base = 'params_active_growth_sweep_Dtheta_regimes_glued'
param_file = project_folder / f'{param_base}.txt'
run_table_file = project_folder / f'{param_base}_runs.csv'

data_folder = project_folder / 'data' / param_base
figure_folder = project_folder / 'figures' / param_base
summary_folder = project_folder / 'analysis_summaries' / param_base

for folder in [data_folder, figure_folder, summary_folder]:
    folder.mkdir(parents=True, exist_ok=True)

compile_first = False
run_simulations = False
skip_existing_outputs = True
selected_run_ids = None
max_parallel = 25

print('Project folder:', project_folder)
print('C file:', c_file)
print('Executable:', exe_file)
print('Parameter file:', param_file)
print('Run table:', run_table_file)
print('Data folder:', data_folder)
print('Figure folder:', figure_folder)

if not c_file.exists():
    raise FileNotFoundError(f'Cannot find {c_file}. Put abm_front_propagation_glued.c in this folder.')

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Sweep parameters

In [ ]:
Lx = 80.0
Ly = 2.0
T = 1000.0
warmup_T = 200.0

R_inter = 0.05
isolation_buffer_factor = 50.0
Ns = 50
rho0 = 750.0
rho_sat = -1.0

v0 = 0.012
Dtheta_values = np.array([0.002, 0.5, 5.0], dtype=float)

q0 = 0.15
r_values = np.array([0.10, 0.20, 0.35, 0.50, 0.70], dtype=float)
p0_values = q0 + r_values

if np.any(p0_values > 1.0):
    raise ValueError('Some p0 values exceed 1. Reduce r_values or q0.')

Dr = 0.00006325
dt = 0.001

seed_values = np.array([69069, 15015015, 123458], dtype=int)

x_init_min = 0.5 * Lx - 0.25
x_init_max = 0.5 * Lx + 0.25

front_per_step = int(round(0.2 / dt))
save_per_step = int(round(10.0 / dt))
rho_profile_every_front = 10

nbins_x = 1600

print('v0:', v0)
print('Dtheta values:', Dtheta_values)
print('r values:', r_values)
print('p0 values:', p0_values)
print('seeds:', seed_values)
print('Total planned runs:', len(Dtheta_values) * len(r_values) * len(seed_values))
print('Isolation buffer B:', isolation_buffer_factor * R_inter)

## 2. Theory and timestep checks

In [ ]:
def D_eff_active(v0, Dtheta, Dr):
    return Dr + v0**2 / (2.0 * Dtheta)


def v_fkpp_brownian(r, Dr):
    return 2.0 * np.sqrt(Dr * r)


def v_fkpp_active_eff(r, Dr, v0, Dtheta):
    return 2.0 * np.sqrt(r * D_eff_active(v0, Dtheta, Dr))


def active_dt_limits(v0, Dtheta_values, Dr, R_inter, p0_values, q0, fraction=0.1):
    v0_max = float(abs(v0))
    Dtheta_max = float(np.max(Dtheta_values))
    p0_max = float(np.max(p0_values))
    q0_abs = float(abs(q0))
    limits = {}
    limits['active: v0*dt < 0.1 R_inter'] = np.inf if v0_max <= 0 else fraction * R_inter / v0_max
    limits['translation: sqrt(2 Dr dt) < 0.1 R_inter'] = np.inf if Dr <= 0 else (fraction * R_inter)**2 / (2.0 * Dr)
    limits['angle: sqrt(2 Dtheta_max dt) < 0.1'] = np.inf if Dtheta_max <= 0 else fraction**2 / (2.0 * Dtheta_max)
    limits['birth/death: max(p0,q0)*dt < 0.01'] = 0.01 / max(p0_max, q0_abs)
    return limits

chosen_dt = float(dt)
dt_limits = active_dt_limits(v0, Dtheta_values, Dr, R_inter, p0_values, q0, fraction=0.1)
dt_max = min(dt_limits.values())
limiting_name = min(dt_limits, key=dt_limits.get)

print('dt limits from the C-code 10% conditions:')
for name, value in dt_limits.items():
    print(f'{name:54s}: {value:g}')
print(f'\ndt_max = {dt_max:g}')
print(f'limiting condition: {limiting_name}')
print(f'chosen dt = {chosen_dt:g}')

check_table = pd.DataFrame([
    {'condition': 'v0 * dt', 'value': abs(v0) * chosen_dt, 'limit': 0.1 * R_inter},
    {'condition': 'sqrt(2 Dr dt)', 'value': np.sqrt(2.0 * Dr * chosen_dt), 'limit': 0.1 * R_inter},
    {'condition': 'sqrt(2 Dtheta_max dt)', 'value': np.sqrt(2.0 * np.max(Dtheta_values) * chosen_dt), 'limit': 0.1},
    {'condition': 'max(p0) * dt', 'value': np.max(p0_values) * chosen_dt, 'limit': 0.01},
    {'condition': 'q0 * dt', 'value': q0 * chosen_dt, 'limit': 0.01},
])
check_table['ratio_value_over_limit'] = check_table['value'] / check_table['limit']
display(check_table)

if chosen_dt > dt_max:
    raise ValueError('Chosen dt violates at least one C-code timestep condition.')
else:
    print('Chosen dt passes the C-code timestep checks.')

theory_rows = []
for Dtheta in Dtheta_values:
    for r in r_values:
        Deff = D_eff_active(v0, Dtheta, Dr)
        theory_rows.append({
            'Dtheta': Dtheta,
            'r_edge': r,
            'D_eff': Deff,
            'v_fkpp_brownian': v_fkpp_brownian(r, Dr),
            'v_fkpp_active_eff': v_fkpp_active_eff(r, Dr, v0, Dtheta),
            'slope_v2_vs_r_theory': 4.0 * Deff,
            'persistence_time': 1.0 / Dtheta,
            'persistence_length': v0 / Dtheta,
            'Dtheta_over_r': Dtheta / r,
        })

theory_table = pd.DataFrame(theory_rows)
display(theory_table)

## 3. Write the glued parameter file

In [ ]:
base = dict(
    q0=q0,
    Ns=Ns,
    R_inter=R_inter,
    rho0=rho0,
    v0=v0,
    Dr=Dr,
    dt=dt,
    T=T,
    Lx=Lx,
    Ly=Ly,
    x_init_min=x_init_min,
    x_init_max=x_init_max,
    warmup_T=warmup_T,
    rho_sat=rho_sat,
    nbins_x=nbins_x,
    threshold_frac1=0.2,
    threshold_frac2=0.5,
    threshold_frac3=0.8,
    save_per_step=save_per_step,
    front_per_step=front_per_step,
    rho_profile_every_front=rho_profile_every_front,
    isolation_buffer_factor=isolation_buffer_factor,
)

param_order = [
    'run_id', 'seed',
    'p0', 'q0', 'Ns', 'R_inter', 'rho0',
    'save_per_step', 'front_per_step', 'rho_profile_every_front',
    'isolation_buffer_factor',
    'v0', 'Dr', 'Dtheta', 'dt', 'T',
    'Lx', 'Ly', 'x_init_min', 'x_init_max', 'warmup_T',
    'rho_sat', 'nbins_x', 'threshold_frac1', 'threshold_frac2', 'threshold_frac3',
]

runs = []
run_id = 0
for Dtheta in Dtheta_values:
    for r_edge in r_values:
        for seed in seed_values:
            run = dict(base)
            run.update(run_id=run_id, seed=int(seed), Dtheta=float(Dtheta), p0=float(q0 + r_edge))
            run['r_edge'] = run['p0'] - run['q0']
            run['D_eff'] = D_eff_active(run['v0'], run['Dtheta'], run['Dr'])
            run['v_fkpp_brownian'] = v_fkpp_brownian(run['r_edge'], run['Dr'])
            run['v_fkpp_active_eff'] = v_fkpp_active_eff(run['r_edge'], run['Dr'], run['v0'], run['Dtheta'])
            run['persistence_time'] = 1.0 / run['Dtheta']
            run['persistence_length'] = run['v0'] / run['Dtheta']
            run['Dtheta_over_r'] = run['Dtheta'] / run['r_edge']
            run['isolation_buffer'] = run['isolation_buffer_factor'] * run['R_inter']
            run['code_type'] = 'glued'
            runs.append(run)
            run_id += 1

run_table = pd.DataFrame(runs)
run_table.to_csv(run_table_file, index=False)

with open(param_file, 'w') as f:
    f.write('# Active glued growth-rate sweep for selected Dtheta regimes.\n')
    f.write('# Sweep r = p0 - q0 by changing p0 and keeping q0 fixed.\n')
    f.write(f'# Fixed active speed v0 = {v0:.16g}.\n')
    f.write(f'# Dtheta values = {list(map(float, Dtheta_values))}.\n')
    f.write(f'# r values = {list(map(float, r_values))}.\n')
    f.write('# Uses abm_front_propagation_glued.c with B = 50 R_inter.\n')
    f.write(f'# dt_max from C-code 10 percent conditions = {dt_max:.16g}\n\n')
    for run in runs:
        f.write('[run]\n')
        for key in param_order:
            f.write(f'{key} = {run[key]}\n')
        f.write('\n')

print('Wrote:', param_file)
print('Wrote:', run_table_file)
display(run_table.head(15))

## 4. Compile and run

In [ ]:
if compile_first:
    compile_c_code(
        c_file=c_file,
        exe_file=exe_file,
        flags=['-O2', '-std=c99', '-Wall', '-Wextra'],
    )
else:
    print('compile_first = False')

In [ ]:
if run_simulations:
    results = run_front_simulations(
        exe_file=exe_file,
        param_file=param_file,
        max_parallel=max_parallel,
        selected_run_ids=selected_run_ids,
        data_folder=data_folder,
        skip_existing=skip_existing_outputs,
        required_output='front',
    )
    display(pd.DataFrame(results))
else:
    print('run_simulations = False')

# Analysis

## 5. Fit front speeds

In [ ]:
main_method = 'th_2'
methods_to_check = ['th_1', 'th_2', 'th_3']
methods_all = ['tip', 'quantile', 'th_1', 'th_2', "th_3"]

main_fit_window = (0.70, 0.95)
fit_windows = [(0.60, 0.90), (0.70, 0.90), (0.70, 0.95), (0.80, 0.95)]


def window_label(a, b):
    return f'{a:.2f}-{b:.2f}'


def safe_float_label(x):
    return f'{x:g}'.replace('-', 'm').replace('.', 'p')


def clean_front_data(front_data, exclude_boundary_hit_rows=True):
    n = len(front_data['time'])
    mask = np.isfinite(np.asarray(front_data['time'], dtype=float))
    if exclude_boundary_hit_rows and 'hit_boundary' in front_data:
        mask &= np.asarray(front_data['hit_boundary'], dtype=float) < 0.5
    out = {}
    for key, value in front_data.items():
        arr = np.asarray(value)
        out[key] = arr[mask] if arr.shape == (n,) else value
    return out


def fit_runs_for_window(run_table, f0, f1, methods=methods_all):
    fit_rows = []
    missing = []
    for _, run in run_table.iterrows():
        rid = int(run['run_id'])
        _, front_file, _ = get_run_files(data_folder, param_base, rid)
        front_file = Path(front_file)
        if not front_file.exists():
            missing.append(rid)
            continue
        _, front_data = read_front_file(front_file)
        front_data = clean_front_data(front_data, exclude_boundary_hit_rows=True)
        fit = fit_front_speeds_for_front_data(
            front_data,
            methods=methods,
            fit_start_fraction=f0,
            fit_end_fraction=f1,
        )
        row = run.to_dict()
        row.update(fit)
        row['fit_window'] = window_label(f0, f1)
        row['fit_start_fraction'] = f0
        row['fit_end_fraction'] = f1
        fit_rows.append(row)
    return pd.DataFrame(fit_rows), missing

run_table = pd.read_csv(run_table_file)
fit_df, missing = fit_runs_for_window(run_table, *main_fit_window, methods=methods_all)

print(f'Main window {window_label(*main_fit_window)}: fitted {len(fit_df)} / {len(run_table)} runs')
if missing:
    print('Missing front files:', missing[:20], '...' if len(missing) > 20 else '')

display(fit_df.head())

## 6. Run-level and pooled speed summaries

In [ ]:
def make_run_level_speed_table(fit_df, group_cols, method=main_method):
    keep_cols = ['run_id'] + list(group_cols) + [
        'v_fkpp_brownian', 'v_fkpp_active_eff', 'D_eff',
        'persistence_time', 'persistence_length', 'Dtheta_over_r',
    ]
    keep_cols = [c for c in keep_cols if c in fit_df.columns]
    speed_long = side_speed_long_table(fit_df, method=method, keep_columns=keep_cols)
    speed_long = speed_long.replace([np.inf, -np.inf], np.nan).dropna(subset=['speed'])
    speed_long['speed_over_active_eff'] = speed_long['speed'] / speed_long['v_fkpp_active_eff']
    speed_long['speed_over_brownian_fkpp'] = speed_long['speed'] / speed_long['v_fkpp_brownian']

    run_cols = ['run_id'] + list(group_cols)
    return (
        speed_long
        .groupby(run_cols, as_index=False)
        .agg(
            run_speed=('speed', 'mean'),
            run_speed_over_active_eff=('speed_over_active_eff', 'mean'),
            run_speed_over_brownian_fkpp=('speed_over_brownian_fkpp', 'mean'),
            left_right_std=('speed', 'std'),
            n_front_sides=('speed', 'count'),
            v_fkpp_brownian=('v_fkpp_brownian', 'first'),
            v_fkpp_active_eff=('v_fkpp_active_eff', 'first'),
            D_eff=('D_eff', 'first'),
            persistence_time=('persistence_time', 'first'),
            persistence_length=('persistence_length', 'first'),
            Dtheta_over_r=('Dtheta_over_r', 'first'),
        )
    )


def summarize_run_level(run_level, group_cols):
    summary = (
        run_level
        .groupby(group_cols, as_index=False)
        .agg(
            n_runs=('run_speed', 'count'),
            mean_speed=('run_speed', 'mean'),
            speed_std=('run_speed', 'std'),
            mean_speed_over_active_eff=('run_speed_over_active_eff', 'mean'),
            speed_over_active_eff_std=('run_speed_over_active_eff', 'std'),
            mean_speed_over_brownian_fkpp=('run_speed_over_brownian_fkpp', 'mean'),
            speed_over_brownian_fkpp_std=('run_speed_over_brownian_fkpp', 'std'),
            v_fkpp_brownian=('v_fkpp_brownian', 'first'),
            v_fkpp_active_eff=('v_fkpp_active_eff', 'first'),
            D_eff=('D_eff', 'first'),
            persistence_time=('persistence_time', 'first'),
            persistence_length=('persistence_length', 'first'),
            Dtheta_over_r=('Dtheta_over_r', 'first'),
        )
    )
    summary['speed_sem'] = summary['speed_std'] / np.sqrt(summary['n_runs'])
    summary['speed_over_active_eff_sem'] = summary['speed_over_active_eff_std'] / np.sqrt(summary['n_runs'])
    summary['speed_over_brownian_fkpp_sem'] = summary['speed_over_brownian_fkpp_std'] / np.sqrt(summary['n_runs'])
    summary['v2_measured'] = summary['mean_speed']**2
    summary['v2_measured_err'] = 2.0 * summary['mean_speed'] * summary['speed_sem']
    summary['v2_active_eff'] = summary['v_fkpp_active_eff']**2
    summary['v2_brownian_fkpp'] = summary['v_fkpp_brownian']**2
    summary['relative_error_active_eff'] = summary['mean_speed_over_active_eff'] - 1.0
    return summary.sort_values(group_cols).reset_index(drop=True)

group_cols = ['Dtheta', 'v0', 'Dr', 'r_edge']
run_level_main = make_run_level_speed_table(fit_df, group_cols, method=main_method)
summary = summarize_run_level(run_level_main, group_cols)

run_level_main.to_csv(summary_folder / 'run_level_speeds_main_window.csv', index=False)
summary.to_csv(summary_folder / 'summary_main_window.csv', index=False)

display(summary)

## 7. Main plot: measured speed versus growth rate

In [ ]:
error_column = 'speed_sem'
fig, ax = plt.subplots(figsize=(8.8, 5.4))
r_line = np.linspace(r_values.min(), r_values.max(), 300)

for Dtheta in Dtheta_values:
    sub = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('r_edge')
    if len(sub) == 0:
        continue
    eb = ax.errorbar(
        sub['r_edge'], sub['mean_speed'], yerr=sub[error_column].fillna(0.0),
        marker='o', linestyle='', capsize=3, label=fr'$D_\theta={Dtheta:g}$ simulation'
    )
    color = eb[0].get_color()
    D_eff = D_eff_active(v0, Dtheta, Dr)
    ax.plot(
        r_line, 2.0 * np.sqrt(D_eff * r_line), linestyle='--', linewidth=1.6,
        color=color, label=fr'$D_\theta={Dtheta:g}$ active FKPP'
    )

ax.set_xlabel(r'$r=p_0-q_0$')
ax.set_ylabel(r'$v_{\rm front}$')
ax.set_title(fr'Active growth sweep, $v_0={v0:g}$, glued $B=50R_{{inter}}$, method={main_method}')
ax.legend(fontsize=10, loc='center left', bbox_to_anchor=(1.02, 0.5), frameon=False)
fig.tight_layout(rect=[0.0, 0.0, 0.76, 1.0])

save_path = figure_folder / 'active_growth_velocity_vs_r_with_active_fkpp.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 8. Main theory test: $v_{\rm front}^2$ versus $r$

In [ ]:
v2_fit_rows = []

for Dtheta in Dtheta_values:
    sub = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('r_edge').copy()
    if len(sub) < 2:
        continue
    x = sub['r_edge'].to_numpy()
    y = sub['v2_measured'].to_numpy()
    yerr = sub['v2_measured_err'].fillna(0.0).to_numpy()
    valid = np.isfinite(x) & np.isfinite(y)
    x_fit = x[valid]
    y_fit = y[valid]
    yerr_fit = yerr[valid]

    slope, intercept = np.polyfit(x_fit, y_fit, 1)
    y_pred = slope * x_fit + intercept
    residuals = y_fit - y_pred
    rmse = np.sqrt(np.mean(residuals**2))
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_fit - np.mean(y_fit))**2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else np.nan

    ok_w = np.isfinite(yerr_fit) & (yerr_fit > 0)
    if np.sum(ok_w) >= 2:
        slope_w, intercept_w = np.polyfit(x_fit[ok_w], y_fit[ok_w], 1, w=1.0 / yerr_fit[ok_w])
    else:
        slope_w, intercept_w = np.nan, np.nan

    D_eff = D_eff_active(v0, Dtheta, Dr)
    theory_slope = 4.0 * D_eff
    v2_fit_rows.append({
        'Dtheta': Dtheta,
        'D_eff': D_eff,
        'slope_unweighted': slope,
        'intercept_unweighted': intercept,
        'slope_weighted': slope_w,
        'intercept_weighted': intercept_w,
        'theory_slope_4Deff': theory_slope,
        'slope_over_theory': slope / theory_slope,
        'weighted_slope_over_theory': slope_w / theory_slope if np.isfinite(slope_w) else np.nan,
        'rmse': rmse,
        'R2': r2,
        'n_points': int(valid.sum()),
    })

    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    ax.errorbar(x_fit, y_fit, yerr=yerr_fit, marker='o', linestyle='', capsize=3, label='simulation')
    r_line = np.linspace(x_fit.min(), x_fit.max(), 300)
    ax.plot(r_line, slope * r_line + intercept, '--', linewidth=2.0, label=fr'fit: slope={slope:.3g}')
    if np.isfinite(slope_w):
        ax.plot(r_line, slope_w * r_line + intercept_w, ':', linewidth=2.4, label=fr'weighted fit: slope={slope_w:.3g}')
    ax.plot(r_line, theory_slope * r_line, '-', linewidth=1.5, label=fr'theory: $4D_{{eff}}={theory_slope:.3g}$')
    ax.axhline(0.0, linestyle=':', linewidth=1.0)
    ax.set_xlabel(r'$r=p_0-q_0$')
    ax.set_ylabel(r'$v_{\rm front}^2$')
    ax.set_title(fr'$D_\theta={Dtheta:g}$, $v_0={v0:g}$')
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_path = figure_folder / f'active_growth_v2_vs_r_fit_Dtheta_{safe_float_label(Dtheta)}.png'
    fig.savefig(save_path, dpi=170, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

v2_fit_df = pd.DataFrame(v2_fit_rows)
display(v2_fit_df)

## 9. Summary of fitted slopes

In [ ]:
fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.plot(v2_fit_df['Dtheta'], v2_fit_df['slope_unweighted'], 'o-', label='unweighted fitted slope')
ax.plot(v2_fit_df['Dtheta'], v2_fit_df['slope_weighted'], 's--', label='weighted fitted slope')
ax.plot(v2_fit_df['Dtheta'], v2_fit_df['theory_slope_4Deff'], 'k:', linewidth=2.0, label=r'theory $4D_{eff}$')
ax.set_xscale('log')
ax.set_xlabel(r'$D_\theta$')
ax.set_ylabel(r'slope of $v_{\rm front}^2$ vs $r$')
ax.set_title(fr'Growth-sweep slope test, $v_0={v0:g}$')
ax.legend(fontsize=10)
fig.tight_layout()
save_path = figure_folder / 'active_growth_v2_slope_vs_Dtheta.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

fig, ax = plt.subplots(figsize=(7.6, 5.0))
ax.plot(v2_fit_df['Dtheta'], v2_fit_df['slope_over_theory'], 'o-', label='unweighted slope / theory')
ax.plot(v2_fit_df['Dtheta'], v2_fit_df['weighted_slope_over_theory'], 's--', label='weighted slope / theory')
ax.axhline(1.0, linestyle=':', linewidth=1.5)
ax.set_xscale('log')
ax.set_xlabel(r'$D_\theta$')
ax.set_ylabel(r'fitted slope / $4D_{eff}$')
ax.set_title('Fraction of active-effective FKPP slope recovered')
ax.legend(fontsize=10)
fig.tight_layout()
save_path = figure_folder / 'active_growth_v2_slope_ratio_vs_Dtheta.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 10. Ratio plot: $v_{\rm front}/v_{\rm eff}$ versus $r$

In [ ]:
fig, ax = plt.subplots(figsize=(8.2, 5.0))
for Dtheta in Dtheta_values:
    sub = summary[np.isclose(summary['Dtheta'], Dtheta)].sort_values('r_edge').copy()
    if len(sub) == 0:
        continue
    ax.errorbar(
        sub['r_edge'], sub['mean_speed_over_active_eff'],
        yerr=sub['speed_over_active_eff_sem'].fillna(0.0),
        marker='o', linestyle='-', capsize=3, label=fr'$D_\theta={Dtheta:g}$'
    )
ax.axhline(1.0, linestyle='--', linewidth=1.2)
ax.set_xlabel(r'$r=p_0-q_0$')
ax.set_ylabel(r'$v_{\rm front}/v_{\rm eff}$')
ax.set_title(fr'Active-effective FKPP ratio, $v_0={v0:g}$')
ax.legend(fontsize=11)
fig.tight_layout()
save_path = figure_folder / 'active_growth_v_over_active_eff_vs_r.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 11. Left-right asymmetry diagnostic

In [ ]:
keep_cols = ['run_id', 'Dtheta', 'v0', 'Dr', 'r_edge']
speed_long = side_speed_long_table(fit_df, method=main_method, keep_columns=keep_cols)
side_summary = (
    speed_long
    .groupby(['Dtheta', 'r_edge', 'side'], as_index=False)
    .agg(mean_side_speed=('speed', 'mean'), std_side_speed=('speed', 'std'), n=('speed', 'count'))
)
side_pivot = side_summary.pivot_table(index=['Dtheta', 'r_edge'], columns='side', values='mean_side_speed').reset_index()
side_pivot['asymmetry'] = side_pivot['left'] - side_pivot['right']
side_pivot['mean_lr'] = 0.5 * (side_pivot['left'] + side_pivot['right'])
side_pivot['relative_asymmetry'] = side_pivot['asymmetry'] / side_pivot['mean_lr']
display(side_pivot)

fig, ax = plt.subplots(figsize=(8.2, 5.0))
ax.axhline(0.0, linestyle='--', linewidth=1.2)
for Dtheta in Dtheta_values:
    sub = side_pivot[np.isclose(side_pivot['Dtheta'], Dtheta)].sort_values('r_edge')
    if len(sub) == 0:
        continue
    ax.plot(sub['r_edge'], sub['relative_asymmetry'], marker='o', linestyle='-', label=fr'$D_\theta={Dtheta:g}$')
ax.set_xlabel(r'$r=p_0-q_0$')
ax.set_ylabel(r'$(\bar v_L-\bar v_R)/\bar v$')
ax.set_title(fr'Left-right speed asymmetry, method={main_method}')
ax.legend(fontsize=11)
fig.tight_layout()
save_path = figure_folder / 'active_growth_left_right_asymmetry_vs_r.png'
fig.savefig(save_path, dpi=170, bbox_inches='tight')
print('Saved:', save_path)
plt.show()

## 12. Fit-window sensitivity

In [ ]:
all_fit_rows = []
all_window_summaries = []

for f0, f1 in fit_windows:
    label = window_label(f0, f1)
    fit_df_w, missing_w = fit_runs_for_window(run_table, f0, f1, methods=methods_all)
    print(f'Window {label}: fitted {len(fit_df_w)} / {len(run_table)} runs')
    if len(fit_df_w) == 0:
        continue
    run_level_w = make_run_level_speed_table(fit_df_w, group_cols, method=main_method)
    summary_w = summarize_run_level(run_level_w, group_cols)
    summary_w['fit_window'] = label
    summary_w['fit_start_fraction'] = f0
    summary_w['fit_end_fraction'] = f1
    fit_df_w['fit_window'] = label
    all_fit_rows.append(fit_df_w)
    all_window_summaries.append(summary_w)

window_summary = pd.concat(all_window_summaries, ignore_index=True) if len(all_window_summaries) else pd.DataFrame()
all_window_fits = pd.concat(all_fit_rows, ignore_index=True) if len(all_fit_rows) else pd.DataFrame()
window_summary.to_csv(summary_folder / 'fit_window_summary.csv', index=False)
all_window_fits.to_csv(summary_folder / 'fit_window_all_fits.csv', index=False)
display(window_summary.head(20))

In [ ]:
if len(window_summary) == 0:
    print('No fit-window summary to plot yet.')
else:
    nrows = len(fit_windows)
    ncols = len(Dtheta_values)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 2.8 * nrows), sharex=True, sharey=True)
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = np.array([axes])
    elif ncols == 1:
        axes = np.array([[ax] for ax in axes])

    for row_idx, (f0, f1) in enumerate(fit_windows):
        label = window_label(f0, f1)
        for col_idx, Dtheta in enumerate(Dtheta_values):
            ax = axes[row_idx, col_idx]
            sub = window_summary[(window_summary['fit_window'] == label) & np.isclose(window_summary['Dtheta'], Dtheta)].sort_values('r_edge')
            if len(sub) > 0:
                ax.errorbar(
                    sub['r_edge'], sub['mean_speed_over_active_eff'],
                    yerr=sub['speed_over_active_eff_sem'].fillna(0.0),
                    marker='o', linestyle='-', capsize=3
                )
            ax.axhline(1.0, linestyle='--', linewidth=1.0)
            if row_idx == 0:
                ax.set_title(fr'$D_\theta={Dtheta:g}$')
            if col_idx == 0:
                ax.set_ylabel(f'fit {label}\n' + r'$v/v_{eff}$')
            if row_idx == nrows - 1:
                ax.set_xlabel(r'$r$')

    fig.suptitle(fr'Fit-window sensitivity, active growth sweep, $v_0={v0:g}$', y=1.01)
    fig.tight_layout()
    save_path = figure_folder / 'active_growth_fit_window_sensitivity_v_over_eff.png'
    fig.savefig(save_path, dpi=170, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 13. Threshold robustness

In [ ]:
def method_summary_from_fit_df(fit_df, method):
    run_level = make_run_level_speed_table(fit_df, group_cols, method=method)
    out = summarize_run_level(run_level, group_cols)
    out['method'] = method
    return out

threshold_summary = pd.concat([method_summary_from_fit_df(fit_df, method) for method in methods_to_check], ignore_index=True)
threshold_summary.to_csv(summary_folder / 'threshold_summary_main_window.csv', index=False)
display(threshold_summary.head())

for Dtheta in Dtheta_values:
    fig, ax = plt.subplots(figsize=(7.6, 5.0))
    for method in methods_to_check:
        sub = threshold_summary[np.isclose(threshold_summary['Dtheta'], Dtheta) & (threshold_summary['method'] == method)].sort_values('r_edge')
        if len(sub) == 0:
            continue
        ax.errorbar(sub['r_edge'], sub['mean_speed'], yerr=sub['speed_sem'].fillna(0.0), marker='o', linestyle='-', capsize=3, label=method)
    ax.set_xlabel(r'$r=p_0-q_0$')
    ax.set_ylabel(r'$v_{\rm front}$')
    ax.set_title(fr'Threshold robustness, $D_\theta={Dtheta:g}$')
    ax.legend(fontsize=10)
    fig.tight_layout()
    save_path = figure_folder / f'active_growth_threshold_robustness_Dtheta_{safe_float_label(Dtheta)}.png'
    fig.savefig(save_path, dpi=170, bbox_inches='tight')
    print('Saved:', save_path)
    plt.show()

## 14. Compact final table and saves

In [ ]:
summary_path = summary_folder / 'active_growth_summary_main_window.csv'
run_level_path = summary_folder / 'active_growth_run_level_main_window.csv'
v2_fit_path = summary_folder / 'active_growth_v2_vs_r_fit_slopes.csv'
side_path = summary_folder / 'active_growth_left_right_asymmetry.csv'

summary.to_csv(summary_path, index=False)
run_level_main.to_csv(run_level_path, index=False)
v2_fit_df.to_csv(v2_fit_path, index=False)
side_pivot.to_csv(side_path, index=False)

compact_cols = [
    'Dtheta', 'r_edge', 'n_runs', 'mean_speed', 'speed_sem',
    'v_fkpp_active_eff', 'mean_speed_over_active_eff', 'speed_over_active_eff_sem',
    'D_eff', 'persistence_time', 'persistence_length',
]
compact = summary[[c for c in compact_cols if c in summary.columns]].copy()
display(compact)

print('Saved:', summary_path)
print('Saved:', run_level_path)
print('Saved:', v2_fit_path)
print('Saved:', side_path)

## Thesis plots

In [ ]:
# Thesis plots

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from matplotlib.lines import Line2D


thesis_figure_folder = figure_folder / "thesis_final_active_growth"
thesis_figure_folder.mkdir(parents=True, exist_ok=True)


plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 15,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "lines.markersize": 6,
})

def style_axis(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.tick_params(direction="out", length=4, width=1)
    return ax

def save_thesis_figure(fig, name):
    pdf_path = thesis_figure_folder / f"{name}.pdf"
    png_path = thesis_figure_folder / f"{name}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print("Saved:", pdf_path)
    print("Saved:", png_path)


df = summary.copy()

Dtheta_plot_values = [0.002, 0.5, 5.0]

error_column = "v2_measured_err"

if "v2_measured" not in df.columns:
    df["v2_measured"] = df["mean_speed"]**2

if "v2_measured_err" not in df.columns:
    df["v2_measured_err"] = 2.0 * df["mean_speed"] * df["speed_sem"]


fig, axes = plt.subplots(
    3, 1,
    figsize=(7.2, 10.0),
    sharex=True,
    sharey=False,
)

axes = np.atleast_1d(axes)

for i, (ax, Dtheta) in enumerate(zip(axes, Dtheta_plot_values)):

    sub = (
        df[np.isclose(df["Dtheta"], Dtheta)]
        .sort_values("r_edge")
        .copy()
    )

    if len(sub) == 0:
        ax.text(
            0.5, 0.5,
            fr"No data for $D_\theta={Dtheta:g}$",
            transform=ax.transAxes,
            ha="center",
            va="center",
        )
        continue

    r_data = sub["r_edge"].to_numpy()
    v2_data = sub["v2_measured"].to_numpy()
    v2_err = sub["v2_measured_err"].fillna(0.0).to_numpy()

    v0_panel = float(sub["v0"].iloc[0])
    Dr_panel = float(sub["Dr"].iloc[0])

    D_eff_panel = D_eff_active(v0_panel, Dtheta, Dr_panel)
    theory_slope = 4.0 * D_eff_panel

    r_line = np.linspace(0.0, 1.04 * np.nanmax(r_data), 300)
    v2_theory = theory_slope * r_line

    ax.errorbar(
        r_data,
        v2_data,
        yerr=v2_err,
        marker="o",
        linestyle="-",
        linewidth=1.8,
        capsize=3,
        color="blue",
    )

    ax.plot(
        r_line,
        v2_theory,
        linestyle="--",
        linewidth=1.8,
        color="red",
    )

    panel_label = chr(ord("a") + i)

    ax.text(
        0.04,
        0.90,
        fr"({panel_label}) $D_\theta={Dtheta:g}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax.set_ylabel(r"$v_{\rm front}^2$")

    style_axis(ax, nbins=4)

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-2, 2))
    ax.yaxis.set_major_formatter(formatter)

    ymax = max(
        np.nanmax(v2_data + v2_err),
        np.nanmax(v2_theory),
    )
    
    if i == 0:
        ax.set_ylim(-5.0e-3, 1.08 * ymax)
    else:
        ax.set_ylim(0.0, 1.08 * ymax)

axes[-1].set_xlabel(r"$r=p_0-q_0$")


simulation_handle = Line2D(
    [0], [0],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="blue",
    label="simulation",
)

theory_handle = Line2D(
    [0], [0],
    linestyle="--",
    linewidth=1.8,
    color="red",
    label=r"effective-FKPP prediction",
)

fig.legend(
    handles=[simulation_handle, theory_handle],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    frameon=False,
    handlelength=2.0,
    columnspacing=1.8,
)

fig.tight_layout(rect=[0.0, 0.055, 1.0, 1.0], h_pad=1.3)

save_thesis_figure(fig, "active_growth_v2_vs_r_three_regimes")
plt.show()

In [ ]:
# Thesis plots

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator, ScalarFormatter
from matplotlib.lines import Line2D


thesis_figure_folder = figure_folder / "thesis_final_active_growth"
thesis_figure_folder.mkdir(parents=True, exist_ok=True)


plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 15,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "lines.markersize": 6,
})

def style_axis(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.tick_params(direction="out", length=4, width=1)
    return ax

def save_thesis_figure(fig, name):
    pdf_path = thesis_figure_folder / f"{name}.pdf"
    png_path = thesis_figure_folder / f"{name}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print("Saved:", pdf_path)
    print("Saved:", png_path)


df = summary.copy()

Dtheta_plot_values = [0.002, 0.5, 5.0]

if "v2_measured" not in df.columns:
    df["v2_measured"] = df["mean_speed"]**2

if "v2_measured_err" not in df.columns:
    df["v2_measured_err"] = 2.0 * df["mean_speed"] * df["speed_sem"]


fig, axes = plt.subplots(
    3, 2,
    figsize=(12.0, 10.0),
    sharex=True,
    sharey=False,
    gridspec_kw={
        "width_ratios": [1.15, 1.0],
        "wspace": 0.32,
        "hspace": 0.28,
    },
)

for i, Dtheta in enumerate(Dtheta_plot_values):

    ax_v2 = axes[i, 0]
    ax_ratio = axes[i, 1]

    sub = (
        df[np.isclose(df["Dtheta"], Dtheta)]
        .sort_values("r_edge")
        .copy()
    )

    if len(sub) == 0:
        ax_v2.text(
            0.5, 0.5,
            fr"No data for $D_\theta={Dtheta:g}$",
            transform=ax_v2.transAxes,
            ha="center",
            va="center",
        )
        ax_ratio.text(
            0.5, 0.5,
            fr"No data for $D_\theta={Dtheta:g}$",
            transform=ax_ratio.transAxes,
            ha="center",
            va="center",
        )
        continue

    r_data = sub["r_edge"].to_numpy(dtype=float)
    v_data = sub["mean_speed"].to_numpy(dtype=float)
    v_err = sub["speed_sem"].fillna(0.0).to_numpy(dtype=float)

    v2_data = sub["v2_measured"].to_numpy(dtype=float)
    v2_err = sub["v2_measured_err"].fillna(0.0).to_numpy(dtype=float)

    v0_panel = float(sub["v0"].iloc[0])
    Dr_panel = float(sub["Dr"].iloc[0])

    D_eff_panel = D_eff_active(v0_panel, Dtheta, Dr_panel)
    theory_slope = 4.0 * D_eff_panel

    r_line = np.linspace(0.0, 1.04 * np.nanmax(r_data), 300)
    v2_theory = theory_slope * r_line

    v_eff_data = 2.0 * np.sqrt(D_eff_panel * r_data)

    ratio_data = v_data / v_eff_data
    ratio_err = v_err / v_eff_data


    ax_v2.errorbar(
        r_data,
        v2_data,
        yerr=v2_err,
        marker="o",
        linestyle="-",
        linewidth=1.8,
        capsize=3,
        capthick=1.0,
        elinewidth=1.0,
        color="blue",
        ecolor="blue",
        zorder=3,
    )

    ax_v2.plot(
        r_line,
        v2_theory,
        linestyle="--",
        linewidth=1.8,
        color="red",
        zorder=2,
    )

    panel_label = chr(ord("a") + i)

    ax_v2.text(
        0.04,
        0.90,
        fr"({panel_label}) $D_\theta={Dtheta:g}$",
        transform=ax_v2.transAxes,
        ha="left",
        va="top",
    )

    ax_v2.set_ylabel(r"$v_{\rm front}^2$")

    formatter = ScalarFormatter(useMathText=True)
    formatter.set_powerlimits((-2, 2))
    ax_v2.yaxis.set_major_formatter(formatter)

    ymax_v2 = max(
        np.nanmax(v2_data + v2_err),
        np.nanmax(v2_theory),
    )

    if i == 0:
        ax_v2.set_ylim(-5.0e-3, 1.08 * ymax_v2)
    else:
        ax_v2.set_ylim(0.0, 1.08 * ymax_v2)

    style_axis(ax_v2, nbins=4)


    ax_ratio.errorbar(
        r_data,
        ratio_data,
        yerr=ratio_err,
        marker="o",
        linestyle="-",
        linewidth=1.8,
        capsize=3,
        capthick=1.0,
        elinewidth=1.0,
        color="blue",
        ecolor="blue",
        zorder=3,
    )

    ax_ratio.axhline(
        1.0,
        linestyle="--",
        linewidth=1.8,
        color="red",
        zorder=2,
    )

    ax_ratio.set_ylabel(r"$v_{\rm front}/v_{\rm FKPP}^{\rm eff}$")

    ymax_ratio = max(1.05, np.nanmax(ratio_data + ratio_err))
    ax_ratio.set_ylim(0.0, 1.12 * ymax_ratio)

    style_axis(ax_ratio, nbins=4)

axes[-1, 0].set_xlabel(r"$r=p_0-q_0$")
axes[-1, 1].set_xlabel(r"$r=p_0-q_0$")


simulation_handle = axes[0, 0].errorbar(
    [np.nan],
    [np.nan],
    yerr=[1.0],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    capsize=3,
    capthick=1.0,
    elinewidth=1.0,
    color="blue",
    ecolor="blue",
    label="simulation",
)

theory_handle = Line2D(
    [0], [0],
    linestyle="--",
    linewidth=1.8,
    color="red",
    label=r"effective-FKPP reference",
)

fig.legend(
    handles=[simulation_handle, theory_handle],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.01),
    ncol=2,
    frameon=False,
    handlelength=2.0,
    columnspacing=1.8,
    fontsize = 20,
)



save_thesis_figure(fig, "active_growth_v2_and_ratio_vs_r_three_regimes")
plt.show()